# Derivatives
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** a derivative as the instantaneous rate of change at a single point on a curve
- **Explain** why the derivative is zero at a minimum or maximum and what that means for finding the best model
- **Interpret** the derivative as the slope of the tangent line to a curve at a given point

> **Tip:** Start by dragging the **x-position slider** along the curve and watch the tangent line rotate and change slope. Pay special attention to the moment the tangent line is perfectly flat — that is where the derivative equals zero.

---
## How we got here

In *02: Limits & Continuity* we learned that a limit describes what a function approaches. The derivative uses exactly that idea to answer a more concrete question: how fast is this function changing at this exact point? It is the slope of the curve at a single location, computed by taking a limit of the slope between two nearby points as those points get infinitely close.

---
## Why this matters for data science

Finding the minimum of a loss function is what model training is. The minimum is the point where the derivative equals zero: the function is neither going up nor going down. In linear regression, you can find this point analytically by setting the derivative of MSE to zero and solving for the weights. In more complex models, you use gradient descent to find it step by step. Either way, the derivative is the compass that points you toward the solution.

---
## Try it yourself

In [2]:
x_range = np.linspace(-2.5, 2.5, 400)

def f(x):  return x**3 - 3*x + 1
def df(x): return 3*x**2 - 3

out_fn  = Output()
out_dfn = Output()

x_slider = FloatSlider(value=0.0, min=-2.3, max=2.3, step=0.05,
    description="x position:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

def render(x_val):
    y_val  = f(x_val)
    dy_val = df(x_val)

    x_tan = np.array([x_val - 1.2, x_val + 1.2])
    y_tan = dy_val * (x_tan - x_val) + y_val

    if dy_val > 0.05:
        slope_desc = "positive — f is increasing here"
    elif dy_val < -0.05:
        slope_desc = "negative — f is decreasing here"
    else:
        slope_desc = "≈ zero — local min or max"

    traces1 = [
        go.Scatter(x=x_range, y=f(x_range), mode="lines",
            line=dict(color=PALETTE["muted"], width=2.5),
            name="f(x) = x³ − 3x + 1"),
        go.Scatter(x=x_tan, y=y_tan, mode="lines",
            line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
            name=f"Tangent  slope = {dy_val:.3f}"),
        go.Scatter(x=[x_val], y=[y_val], mode="markers",
            marker=dict(color=PALETTE["primary"], size=12),
            name=f"x = {x_val:.2f},  f(x) = {y_val:.3f}"),
    ]
    layout1 = base_layout(
        title=f"x = {x_val:.2f}  |  f(x) = {y_val:.3f}  |  f′(x) = {dy_val:.3f}  |  slope is {slope_desc}",
        xaxis_title="x", yaxis_title="f(x)")
    layout1.update(xaxis=dict(range=[-2.5, 2.5]), yaxis=dict(range=[-4, 6]))
    with out_fn:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces1, layout=layout1)))

    traces2 = [
        go.Scatter(x=x_range, y=df(x_range), mode="lines",
            line=dict(color=PALETTE["accent"], width=2.5),
            name="f′(x) = 3x² − 3"),
        go.Scatter(x=[-2.5, 2.5], y=[0, 0], mode="lines",
            line=dict(color=PALETTE["muted"], width=1, dash="dot"), showlegend=False),
        go.Scatter(x=[x_val], y=[dy_val], mode="markers",
            marker=dict(color=PALETTE["primary"], size=12),
            name=f"f′({x_val:.2f}) = {dy_val:.3f}"),
    ]
    layout2 = base_layout(
        title=f"f′(x) = 3x² − 3  |  At x = {x_val:.2f}: f′(x) = {dy_val:.3f}",
        xaxis_title="x", yaxis_title="f′(x)")
    layout2.update(xaxis=dict(range=[-2.5, 2.5]), yaxis=dict(range=[-4, 6]))
    with out_dfn:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces2, layout=layout2)))

def on_change(change): render(x_slider.value)
x_slider.observe(on_change, names="value")
display(VBox([x_slider, out_fn, out_dfn]))
render(0.0)

---
### Where does the derivative come from?

The tangent line above shows *what* the derivative is. This widget shows *where it comes from*.

Draw a line through two points on the curve — (x, f(x)) and (x+h, f(x+h)). That line is called the **secant line** and its slope is the **difference quotient**:

$$\text{slope} = \frac{f(x+h) - f(x)}{h}$$

Now shrink h toward zero. Watch what happens to the secant line.

In [ ]:
x_sq = np.linspace(-2.5, 3.5, 500)
out_sq = Output()

x_base_sl = FloatSlider(value=-1.5, min=-2.0, max=1.0, step=0.1,
    description="x (base):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"))

h_sl = FloatSlider(value=1.5, min=0.01, max=2.0, step=0.01,
    description="h (step):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"))

def render_sq(x0, h):
    xh = x0 + h
    y0, yh = f(x0), f(xh)

    sec_slope  = (yh - y0) / h
    true_slope = df(x0)

    x_sec = np.array([x0 - 0.4, xh + 0.4])
    y_sec = y0 + sec_slope * (x_sec - x0)

    y_bar = -4.5

    traces = [
        go.Scatter(x=x_sq, y=f(x_sq), mode="lines",
            line=dict(color=PALETTE["muted"], width=2.5),
            name="f(x) = x³ − 3x + 1"),
        go.Scatter(x=x_sec, y=y_sec, mode="lines",
            line=dict(color=PALETTE["secondary"], width=2.5),
            name=f"Secant line  slope = {sec_slope:.4f}"),
        # Vertical guides from bar to each point
        go.Scatter(x=[x0, x0], y=[y_bar, y0], mode="lines",
            line=dict(color=PALETTE["accent"], width=1.5, dash="dash"),
            showlegend=False),
        go.Scatter(x=[xh, xh], y=[y_bar, yh], mode="lines",
            line=dict(color=PALETTE["accent"], width=1.5, dash="dash"),
            showlegend=False),
        # h bar along bottom
        go.Scatter(x=[x0, xh], y=[y_bar, y_bar], mode="lines",
            line=dict(color=PALETTE["accent"], width=3),
            showlegend=False),
        go.Scatter(x=[(x0 + xh) / 2], y=[y_bar + 0.35], mode="text",
            text=[f"h = {h:.2f}"],
            textfont=dict(color=PALETTE["accent"], size=13),
            showlegend=False),
        # The two points
        go.Scatter(x=[x0], y=[y0], mode="markers",
            marker=dict(color=PALETTE["primary"], size=12),
            name=f"(x, f(x)) = ({x0:.2f}, {y0:.3f})"),
        go.Scatter(x=[xh], y=[yh], mode="markers",
            marker=dict(color=PALETTE["primary"], size=12, symbol="diamond"),
            name=f"(x+h, f(x+h)) = ({xh:.2f}, {yh:.3f})"),
    ]

    diff  = abs(sec_slope - true_slope)
    title = (f"Secant slope = {sec_slope:.4f}  |  "
             f"True f′(x) = {true_slope:.4f}  |  "
             f"Difference = {diff:.4f}")
    layout = base_layout(title=title, xaxis_title="x", yaxis_title="f(x)")
    layout.update(
        xaxis=dict(range=[min(x0 - 0.6, -2.5), max(xh + 0.6, 2.5)]),
        yaxis=dict(range=[-5.5, 7]))
    with out_sq:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

def on_change_sq(change): render_sq(x_base_sl.value, h_sl.value)
x_base_sl.observe(on_change_sq, names="value")
h_sl.observe(on_change_sq, names="value")
display(VBox([x_base_sl, h_sl, out_sq]))
render_sq(-1.5, 1.5)

---
## What's happening?

The derivative of a function at a point is the slope of the line that just touches the curve at that point without crossing it. We call that line the tangent line. Formally, it is defined as the limit of the slope between two points as the second point gets infinitely close to the first.

The slope between two points (x, f(x)) and (x+h, f(x+h)) is:

$$\text{slope} = \frac{f(x+h) - f(x)}{h}$$

As h shrinks toward zero, this becomes the derivative:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

We read f'(x) as "f prime of x." It is itself a function: it gives the slope of f at every point x.

### Common derivatives you will see in ML

| Function f(x) | Its derivative f'(x) | What zero derivative means | ML significance |
|--------------|---------------------|---------------------------|----------------|
| f(x) = x² | f'(x) = 2x | Zero at x = 0: the minimum of the parabola | MSE loss has its minimum at the best-fit weights |
| f(x) = eˣ | f'(x) = eˣ | Never zero: always growing | Exponential activations never have flat spots |
| f(x) = ln(x) | f'(x) = 1/x | Never zero for x > 0 | Log loss gradient is never exactly zero |
| Sigmoid σ(x) | σ(x)(1 − σ(x)) | Zero at ±∞: saturated neuron | Vanishing gradient in deep networks |

### Where the derivative equals zero is where extremes live

When f'(x) = 0, the tangent line is horizontal. The function is momentarily flat. This happens at every minimum, maximum, and saddle point. Finding these zero-crossing points of the derivative is how we find the best parameters for a model.

Drag the slider to the point where the tangent line in the widget is flat and confirm the derivative readout shows zero.

---
## Real-world example: The derivative of MSE tells linear regression where to go

Mean squared error for a linear model with one weight w is a parabola in w. The derivative of that parabola is a straight line that crosses zero at exactly the optimal w. The chart shows this: the MSE curve and its derivative, with the zero-crossing of the derivative marking the best weight.

Notice:

- **Notice:** The MSE curve is always bowl-shaped (convex) with exactly one minimum; this is a special property of linear regression that makes it easy to optimize
- **Notice:** The derivative curve is a straight line that equals zero at exactly the same w where the MSE is minimized; this is not a coincidence, it is the definition of a minimum
- **Notice:** To the left of the minimum the derivative is negative (MSE is decreasing), and to the right it is positive (MSE is increasing); this sign tells gradient descent which direction to step

> **Discussion question:** For linear regression with one weight, you can find the exact optimal w by setting the derivative to zero and solving. For a neural network with a million weights, why is that approach impossible, and what do you do instead?

In [4]:
# ── Derivative of MSE shows where the optimal weight lives ────────────────────
# Simulate 30 data points with true weight w*=2.5
np.random.seed(42)
x_data = np.random.uniform(0, 5, 30)
y_data = 2.5 * x_data + np.random.normal(0, 1.5, 30)

# MSE as a function of weight w (with optimal intercept absorbed)
w_range = np.linspace(-1, 6, 300)
mse     = np.array([np.mean((y_data - w * x_data) ** 2) for w in w_range])
dmse    = np.gradient(mse, w_range)   # numerical derivative of MSE

w_opt   = w_range[np.argmin(mse)]

fig = make_subplots(rows=2, cols=1,
    subplot_titles=[f"MSE Loss as a Function of Weight w  (minimum at w ≈ {w_opt:.2f})",
                    "Derivative of MSE — crosses zero at the optimal weight"])
fig.add_trace(go.Scatter(x=w_range, y=mse, mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5), name="MSE(w)"), row=1, col=1)
fig.add_vline(x=w_opt, line_dash="dash", line_color=PALETTE["secondary"],
              annotation_text=f"w* = {w_opt:.2f}", row=1, col=1)

fig.add_trace(go.Scatter(x=w_range, y=dmse, mode="lines",
    line=dict(color=PALETTE["secondary"], width=2.5), name="dMSE/dw"), row=2, col=1)
fig.add_hline(y=0, line_dash="dot", line_color=PALETTE["muted"], row=2, col=1)
fig.add_vline(x=w_opt, line_dash="dash", line_color=PALETTE["secondary"], row=2, col=1)

fig.update_layout(paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
                  title=dict(text="MSE and Its Derivative — Zero Derivative Marks the Optimal Weight",
                             font=dict(size=FONT["size_title"])),
                  height=500, showlegend=False)
fig.update_xaxes(title_text="Weight w", gridcolor="#DEE2E6")
fig.update_yaxes(gridcolor="#DEE2E6")
fig.show()

### Derivative rules every ML practitioner encounters

| Rule | What it says | Why it appears in ML |
|------|-------------|---------------------|
| Power rule: d/dx xⁿ = nxⁿ⁻¹ | Bring down the exponent, reduce it by one | Derivative of MSE (which involves x²) |
| Constant rule: d/dx c = 0 | Constants vanish | Bias terms drop out cleanly |
| Sum rule: (f+g)' = f' + g' | Differentiate term by term | Derivative of a sum of losses over a batch |
| Chain rule: (f∘g)' = f'(g)·g' | Multiply the outer and inner derivatives | Backpropagation through composed layers |
| Product rule: (fg)' = f'g + fg' | The product expansion | Appears in regularization gradient terms |

---
## Key takeaway

> **The derivative is a function that maps each point on a curve to its instantaneous slope; where this derivative function equals zero, the curve is flat, identifying the critical points where the minimum or maximum of a loss function occurs**

---
*Next up: The Chain Rule — how to differentiate functions that are built from other functions, and why that is the mathematical engine behind training every neural network*